### Load Data

In [1]:
import polars as pl
import duckdb

df = pl.read_parquet("data/uc-faculty.parquet")


### Basic Data Information


In [2]:
df.describe()

statistic,year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay
str,f64,str,str,str,str,str,f64,f64,f64,f64
"""count""",3.660661e6,"""3660661""","""3660661""","""3660660""","""3660657""","""3660320""",3.660661e6,3.660661e6,3.660661e6,3.660661e6
"""null_count""",0.0,"""0""","""0""","""1""","""4""","""341""",0.0,0.0,0.0,0.0
"""mean""",2018.764224,null,null,null,null,null,56705.176119,48504.964208,847.00437,7353.209968
"""std""",3.458144,null,null,null,null,null,78820.283732,55530.470437,3942.074732,39005.584722
"""min""",2013.0,null,""" """,""" ""","""9964 - no description found""",null,1.0,-385300.0,-49939.0,-345255.0
"""25%""",2016.0,null,null,null,null,null,5166.0,4508.0,0.0,0.0
"""50%""",2019.0,null,null,null,null,null,34225.0,31786.0,0.0,0.0
"""75%""",2022.0,null,null,null,null,null,76860.0,71906.0,0.0,2500.0
"""max""",2024.0,null,"""zyzik ""","""ângela""","""zone mech""",null,7.061667e6,1.965054e6,298933.0,6.761667e6


In [3]:
df.head()

year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay
i16,cat,str,str,str,cat,i32,i32,i32,i32
2024,"""asucla""","""abbassi""","""pouria""","""dir exec""","""executive director""",390489,356085,0,34405
2024,"""asucla""","""mehdian""","""kamran""","""dir""","""systems development""",247343,236450,0,10893
2024,"""asucla""","""baker""","""donna""","""dir""","""financial planning & analysis""",243072,232372,0,10700
2024,"""asucla""","""moyer""","""michelle""","""dir""","""human resources""",241336,231051,0,10285
2024,"""asucla""","""bolton""","""cynthia""","""dir""","""rest operations""",182084,174324,0,7760


### Basic Data Filters
Table name for SQL statements is faculty_salaries

In [4]:
# Min base_pay must be higher than minimum wage full time
# To try and only have educational faculty we fuzzy match on prof, lect, and adj.
duckdb.connect()
df = duckdb.sql("""
    SELECT *
    FROM df
    WHERE 
        title_name ILIKE '%prof%'
        OR title_name ILIKE '%lect%'
        OR title_name ILIKE '%adj%'
        AND base_pay >= 35152
    """).df()

duckdb.sql("""
    CREATE TABLE faculty_salaries AS
        SELECT *
        FROM df
           """)




Data info after filter

In [5]:
df.describe()

,year,gross_pay,base_pay,overtime_pay,other_pay
count,335596.000000,3.355960e+05,335596.000000,335596.000000,3.355960e+05
mean,2018.917523,1.614787e+05,109931.468149,95.967300,5.145131e+04
std,3.428940,1.564654e+05,80679.204604,1218.756688,1.062002e+05
min,2013.000000,1.000000e+00,-385300.000000,-8585.000000,-3.452550e+05
25%,2016.000000,5.029775e+04,42808.750000,0.000000,0.000000e+00
50%,2019.000000,1.281485e+05,106833.000000,0.000000,9.336000e+03
75%,2022.000000,2.236278e+05,156658.000000,0.000000,5.556700e+04
max,2024.000000,3.974061e+06,838117.000000,79214.000000,3.695451e+06


In [6]:
df.head()

,year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay
0,2024,asucla,tejeda,steve,electrn,maintenance facilities,79706,73479,5645,583
1,2024,berkeley,malmendier,ulrike,prof-ay-b/e/e,haas core programs,730483,548817,0,181667
2,2024,berkeley,chatman,jennifer,prof-ay-b/e/e,haas core programs,718700,445800,0,272900
3,2024,berkeley,yaghi,omar,prof-ay,dept of chemistry,690298,441667,0,248631
4,2024,berkeley,isacoff,ehud,prof-ay,molecular & cell biology,680880,481058,0,199822


## Department Cleaning

In [7]:
duckdb.sql("""
SELECT DISTINCT department
FROM faculty_salaries
           """)

┌───────────────────────────────┐
│          department           │
│            varchar            │
├───────────────────────────────┤
│ mechanical & aerospace engr   │
│ med: dermatology              │
│ mathematics                   │
│ gendersexuality womensstudies │
│ center for mind & brain       │
│ anthropology                  │
│ dermatology                   │
│ psychology & social behavior  │
│ philosophy                    │
│ hum graduate office           │
│          ·                    │
│          ·                    │
│          ·                    │
│ ghei irvine - opth clinic     │
│ john r lewis hsg/student life │
│ c external relations comm i/o │
│ educational technology srvs   │
│ international house           │
│ critical theory               │
│ it enterprise applications    │
│ air quality research center   │
│ social sciences blue cluster  │
│ internal medicine gme         │
└───────────────────────────────┘
      4469 rows (20 shown)     